# Redução de dimensionalidade em imagens — DIO
**Autor:** Giordani Martins Silva. Código e documentação preparados com auxílio de IA.

Objetivo: transformar RGB em um canal de cinza (0 a 255) e aplicar um limiar para obter preto e branco (0 ou 255). A transformação dos pixels é implementada explicitamente, sem chamar convert('L') nem uma função pronta de threshold.

Execute as células na ordem. Não precisa de GPU. O exemplo padrão é gerado localmente e não depende de download de fotos. É possível enviar sua própria imagem na etapa 3.


In [ ]:
%pip -q install 'Pillow>=10.1,<13'

## 1. Algoritmo
Cinza = arredondamento de 0,299R + 0,587G + 0,114B. A implementação usa inteiros e soma 500 antes de dividir por 1000 para arredondar. Em seguida: cinza >= limiar vira 255; abaixo vira 0. O padrão é 128.

In [ ]:
"""Conversão explícita por pixel: RGB -> cinza -> binária (0 ou 255)."""
import argparse
import json
from pathlib import Path
from PIL import Image, ImageOps


def preparar_rgb(image):
    """Respeita orientação EXIF e compõe transparência sobre fundo branco."""
    image = ImageOps.exif_transpose(image).convert("RGBA")
    background = Image.new("RGBA", image.size, (255, 255, 255, 255))
    return Image.alpha_composite(background, image).convert("RGB")


def converter_cinza(image):
    """Luma aproximada: 0,299R + 0,587G + 0,114B, arredondada para inteiro."""
    if image.mode != "RGB":
        raise ValueError("converter_cinza requer uma imagem RGB; use preparar_rgb.")
    result = Image.new("L", image.size)
    pixels = image.load()
    values = []
    for y in range(image.height):
        for x in range(image.width):
            r, g, b = pixels[x, y]
            values.append((299 * r + 587 * g + 114 * b + 500) // 1000)
    result.putdata(values)
    return result


def binarizar(gray, limiar=128):
    """Cinza >= limiar vira branco (255); abaixo vira preto (0)."""
    if type(limiar) is not int or not 0 <= limiar <= 255:
        raise ValueError("O limiar deve ser um inteiro entre 0 e 255.")
    if gray.mode != "L":
        raise ValueError("binarizar requer uma imagem em modo L.")
    result = Image.new("L", gray.size)
    result.putdata([255 if value >= limiar else 0 for value in gray.tobytes()])
    return result


def montar_comparacao(rgb, gray, binary, limiar):
    from PIL import ImageDraw, ImageFont
    width, height = rgb.size
    margin, title_height = 12, 42
    canvas = Image.new("RGB", (3 * width + 4 * margin, height + title_height + margin), "#f5f5f5")
    draw = ImageDraw.Draw(canvas)
    font = ImageFont.load_default(size=18)
    titles = ["Original RGB", "Cinza: 0 a 255", f"Binaria: limiar {limiar}"]
    for index, (image, title) in enumerate(zip([rgb, gray, binary], titles)):
        x = margin + index * (width + margin)
        draw.text((x, 12), title, fill="#171717", font=font)
        canvas.paste(image.convert("RGB"), (x, title_height))
    return canvas


def processar(input_path, output_dir="resultados", limiar=128):
    output = Path(output_dir)
    input_path = Path(input_path)
    # Valida o limiar antes de ler ou criar arquivos.
    if type(limiar) is not int or not 0 <= limiar <= 255:
        raise ValueError("O limiar deve ser um inteiro entre 0 e 255.")
    with Image.open(input_path) as source:
        rgb = preparar_rgb(source)
    gray = converter_cinza(rgb)
    binary = binarizar(gray, limiar)
    output.mkdir(parents=True, exist_ok=True)
    targets = [output / name for name in ["original.png", "cinza.png", "binaria.png", "comparacao.png", "relatorio.json"]]
    if input_path.resolve() in [p.resolve() for p in targets]:
        raise ValueError("A saída sobrescreveria a entrada; escolha outra pasta.")
    rgb.save(output / "original.png")
    gray.save(output / "cinza.png")
    binary.save(output / "binaria.png")
    montar_comparacao(rgb, gray, binary, limiar).save(output / "comparacao.png")
    values = list(binary.tobytes())
    report = {
        "entrada": input_path.name, "largura": rgb.width, "altura": rgb.height,
        "pixels": rgb.width * rgb.height, "limiar": limiar,
        "formula": "(299*R + 587*G + 114*B + 500) // 1000",
        "regra": "cinza >= limiar: 255; caso contrário: 0",
        "modos": {"original": rgb.mode, "cinza": gray.mode, "binaria": binary.mode},
        "intervalo_cinza": list(gray.getextrema()),
        "valores_binaria": sorted(set(values)),
        "pixels_pretos": values.count(0), "pixels_brancos": values.count(255),
        "dimensoes_preservadas": rgb.size == gray.size == binary.size,
    }
    (output / "relatorio.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    return report


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("imagem", help="Caminho de uma imagem PNG ou JPEG")
    parser.add_argument("--saida", default="resultados")
    parser.add_argument("--limiar", type=int, default=128)
    args = parser.parse_args()
    try:
        report = processar(args.imagem, args.saida, args.limiar)
    except (OSError, ValueError) as error:
        parser.exit(1, f"Erro: {error}\n")
    print(json.dumps(report, ensure_ascii=False, indent=2))



## 2. Gerar uma imagem colorida de exemplo
O gradiente e as faixas de cores permitem observar que cores diferentes podem virar o mesmo nível de cinza ou a mesma classe binária.

In [ ]:
"""Gera uma imagem de teste própria: gradiente RGB e áreas de cores conhecidas."""
from pathlib import Path
from PIL import Image, ImageDraw


def criar_exemplo(path="exemplos/entrada.png"):
    width, height = 384, 256
    image = Image.new("RGB", (width, height))
    image.putdata([(round(255 * x / (width - 1)), round(255 * y / (height - 1)),
                    round(255 * (1 - x / (width - 1))))
                   for y in range(height) for x in range(width)])
    draw = ImageDraw.Draw(image)
    for index, color in enumerate(["red", "lime", "blue", "yellow", "cyan", "magenta", "black", "white"]):
        draw.rectangle((index * 48, 192, (index + 1) * 48 - 1, 255), fill=color)
    target = Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    image.save(target)
    return target


entrada = criar_exemplo()


## 3. Usar imagem própria (opcional)
Mude usar_imagem_propria para True para abrir a seleção de arquivo no Colab. PNG e JPEG são aceitos. Imagens com transparência são compostas sobre branco; a orientação EXIF é respeitada.

In [ ]:
usar_imagem_propria = False
if usar_imagem_propria:
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("Nenhum arquivo foi enviado.")
    entrada = Path(next(iter(uploaded)))
print("Entrada:", entrada)


## 4. Processar e conferir
Experimente 64, 128 e 192. Quanto maior o limiar, menor ou igual é a quantidade de pixels brancos. A igualdade com o limiar é classificada como branco.

In [ ]:
limiar = 128
relatorio = processar(entrada, "resultados", limiar)
print(json.dumps(relatorio, ensure_ascii=False, indent=2))
assert set(relatorio["valores_binaria"]).issubset({0, 255})
assert relatorio["dimensoes_preservadas"]
assert relatorio["pixels_pretos"] + relatorio["pixels_brancos"] == relatorio["pixels"]


## 5. Visualizar os três resultados

In [ ]:
try:
    from IPython.display import display
except ImportError:
    print("Comparação salva em resultados/comparacao.png")
else:
    with Image.open("resultados/comparacao.png") as preview:
        display(preview.copy())


## 6. Baixar os arquivos
O ZIP contém original normalizada, cinza, binária, comparação e relatório JSON.

In [ ]:
import shutil
archive = shutil.make_archive("imagens-processadas", "zip", "resultados")
try:
    from google.colab import files
except ImportError:
    print("Arquivo gerado:", archive)
else:
    files.download(archive)


## O que foi reduzido?
A imagem RGB passa de três canais para um. A binarização reduz os valores possíveis para dois. **Não há redução de largura ou altura**, nem aplicação de PCA. As saídas PNG são salvas em modo L de 8 bits: ter dois valores não significa que o arquivo foi salvo com um bit por pixel.

A fórmula é uma aproximação usual de luma sobre valores RGB, sem linearização de gamma; não equivale a medir luminância física. Um limiar global pode perder detalhes em imagens com iluminação desigual.

O notebook é distribuído sem saídas para nova execução. As evidências executadas localmente estão no repositório. Este material não comprova visualização das aulas nem uma sessão no Colab.
